In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, mean_absolute_percentage_error

# preprocessing
raw_name = "data.csv"
daily_name = "daily.csv"
weekly_name = "weekly.csv"
monthly_name ="monthly.csv"
load_chunksize = 1_000_000

# models
hyperparams = {
    'adaboost': {
        'n_estimators':[150,500,1500],
        'learning_rate':[0.02,0.04,0.08],
    },
    'xgboost': {
        'n_estimators':[150,500,1500],
        'learning_rate':[0.02,0.04,0.08],
        'max_depth':[3,5,8],
        'subsample':[0.6,0.8,1],
        'gamma':[0,0.1,0.25,0.5]
    }
}
ts_splits = 5

print('Setup complete!')


Setup complete!


### Preprocessing

In [2]:
def add_lag_features(df, target_col, lags):
    df = df.sort_values("date").copy()
    
    for lag in lags:
        df[f"{target_col}_lag_{lag}"] = df[target_col].shift(lag)
    
    return df

In [3]:

results = []

for chunk in pd.read_csv(raw_name, chunksize=load_chunksize):

    chunk = chunk.dropna(subset=["onpromotion"])
    chunk["onpromotion"] = chunk["onpromotion"].astype(int)

    # weighted promotion = sales * promotion
    chunk["promo_weight"] = chunk["unit_sales"] * chunk["onpromotion"]

    grouped = chunk.groupby("date").agg(
        sales=("unit_sales", "sum"),
        promo_weight=("promo_weight", "sum")
    ).reset_index()

    results.append(grouped)

print('Dataset loaded!')

daily = pd.concat(results)
# aggregate across chunks
daily = daily.groupby("date").agg(
    sales=("sales", "sum"),
    promo_weight=("promo_weight", "sum")
).reset_index()
daily["promotion"] = daily["promo_weight"] / daily["sales"].replace(0, pd.NA)

daily = daily[["date", "sales", "promotion"]]
daily["date"] = pd.to_datetime(daily["date"])
daily["day_of_week"] = daily["date"].dt.dayofweek
daily["month"] = daily["date"].dt.month
daily["year"] = daily["date"].dt.year

daily = add_lag_features(
    daily,
    target_col="sales",
    lags=[1, 7, 14, 28]
)
daily = daily.dropna()

daily.to_csv(daily_name, index=False)
print('Daily aggregation saved!')


C:\Users\csg12\AppData\Local\Temp\ipykernel_9348\396133160.py:3: DtypeWarning: Columns (0: onpromotion) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(raw_name, chunksize=load_chunksize):


Dataset loaded!
Daily aggregation saved!


In [4]:
# --- WEEKLY AGGREGATION ---
weekly = daily.copy()

# Reconstruct promo_weight to avoid averaging ratios
weekly["promo_weight"] = weekly["promotion"] * weekly["sales"]

weekly = weekly.groupby(pd.Grouper(key="date", freq="W")).agg(
    sales=("sales", "sum"),
    promo_weight=("promo_weight", "sum")
).reset_index()

weekly["promotion"] = weekly["promo_weight"] / weekly["sales"]

weekly = weekly[["date", "sales", "promotion"]]
weekly["month"] = weekly["date"].dt.month
weekly["year"] = weekly["date"].dt.year

weekly = add_lag_features(
    weekly,
    target_col="sales",
    lags=[1, 2, 4, 8]
)
weekly = weekly.dropna()

weekly.to_csv(weekly_name, index=False)
print('Weekly aggregation saved!')

# --- MONTHLY AGGREGATION ---
monthly = daily.copy()

monthly["promo_weight"] = monthly["promotion"] * monthly["sales"]

monthly = monthly.groupby(pd.Grouper(key="date", freq="ME")).agg(
    sales=("sales", "sum"),
    promo_weight=("promo_weight", "sum")
).reset_index()

monthly["promotion"] = monthly["promo_weight"] / monthly["sales"]

monthly = monthly[["date", "sales", "promotion"]]
monthly["month"] = monthly["date"].dt.month
monthly["year"] = monthly["date"].dt.year

monthly = add_lag_features(
    monthly,
    target_col="sales",
    lags=[1, 2, 3, 6]
)
monthly = monthly.dropna()

monthly.to_csv(monthly_name, index=False)
print('Monthly aggregation saved!')


Weekly aggregation saved!
Monthly aggregation saved!


In [5]:
if "daily" not in globals():
    daily = pd.read_csv(daily_name)
    print('Daily aggregation loaded!')

if "weekly" not in globals():
    weekly = pd.read_csv(weekly_name)
    print('Weekly aggregation loaded!')

if "monthly" not in globals():
    monthly = pd.read_csv(monthly_name)
    print('Monthly aggregation loaded!')

print('Data ready!')

Data ready!


### Models

In [6]:
# OLD

ts_split = TimeSeriesSplit(n_splits=ts_splits)
results = []

for data in daily, weekly, monthly:

    split_idx = int(len(data) * 0.8)
    train_data = data.iloc[:split_idx]
    train_X = train_data[train_data.columns.drop(['sales','date']).tolist()]
    train_y = train_data['sales']
    test_data  = data.iloc[split_idx:]
    test_X = test_data[test_data.columns.drop(['sales','date']).tolist()]
    test_y = test_data['sales']

    for model, params in [
        (AdaBoostRegressor(random_state=42),hyperparams['adaboost']),
        (XGBRegressor(objective="reg:squarederror",random_state=42,n_jobs=1),hyperparams['xgboost'])
    ]:

        grid = GridSearchCV(
            estimator=model,
            param_grid=params,
            cv=ts_split,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=1
        )
        grid.fit(train_X, train_y)

        results.append ({
            "best_model": grid.best_estimator_,
            "best_params": grid.best_params_,
            "best_score": grid.best_score_,
            "all_results": grid.cv_results_
        })

for i, r in enumerate(results, start=1):
    model_type = type(r["best_model"]).__name__
    params = r["best_params"]
    score = r["best_score"]

    print(f"\nModel {i}")
    print(f"Type: {model_type}")
    print(f"Best Score: {score}")
    print(f"Best Params: {params}")

Fitting 5 folds for each of 9 candidates, totalling 45 fits
Fitting 5 folds for each of 324 candidates, totalling 1620 fits


KeyboardInterrupt: 

In [7]:
# ---------------------------------------------
# Hyperparameter tuning with TimeSeriesSplit
# ---------------------------------------------

# datasets to iterate over
datasets = {
    'daily': daily,
    'weekly': weekly,
    'monthly': monthly
}

# store best fitted models and metadata
best_models = []

# time-series cross validation
tscv = TimeSeriesSplit(n_splits=ts_splits)

for dataset_name, df in datasets.items():

    print(f"\n===== Processing {dataset_name} dataset =====")

    # features and target
    X = df.drop(columns=['date', 'sales'])
    y = df['sales']

    # -----------------------------
    # AdaBoost
    # -----------------------------
    print(f"Running AdaBoost GridSearchCV for {dataset_name}...")

    ada_model = AdaBoostRegressor(random_state=42)

    ada_search = GridSearchCV(
        estimator=ada_model,
        param_grid=hyperparams['adaboost'],
        cv=tscv,
        scoring='neg_mean_absolute_error',
        n_jobs=-1,
        verbose=1
    )

    ada_search.fit(X, y)

    best_ada = ada_search.best_estimator_

    best_models.append({
        'dataset': dataset_name,
        'model_name': 'AdaBoost',
        'best_model': best_ada,
        'best_params': ada_search.best_params_,
        'best_score': -ada_search.best_score_
    })

    print("Best AdaBoost params:", ada_search.best_params_)
    print("Best AdaBoost MAE:", -ada_search.best_score_)

    # -----------------------------
    # XGBoost
    # -----------------------------
    print(f"\nRunning XGBoost GridSearchCV for {dataset_name}...")

    xgb_model = XGBRegressor(
        objective='reg:squarederror',
        random_state=42
    )

    xgb_search = GridSearchCV(
        estimator=xgb_model,
        param_grid=hyperparams['xgboost'],
        cv=tscv,
        scoring='neg_mean_absolute_error',
        n_jobs=-1,
        verbose=1
    )

    xgb_search.fit(X, y)

    best_xgb = xgb_search.best_estimator_

    best_models.append({
        'dataset': dataset_name,
        'model_name': 'XGBoost',
        'best_model': best_xgb,
        'best_params': xgb_search.best_params_,
        'best_score': -xgb_search.best_score_
    })

    print("Best XGBoost params:", xgb_search.best_params_)
    print("Best XGBoost MAE:", -xgb_search.best_score_)

print("\nHyperparameter tuning complete!")
print(f"Stored {len(best_models)} best models.")


===== Processing daily dataset =====
Running AdaBoost GridSearchCV for daily...
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best AdaBoost params: {'learning_rate': 0.02, 'n_estimators': 150}
Best AdaBoost MAE: 86419.85504793076

Running XGBoost GridSearchCV for daily...
Fitting 5 folds for each of 324 candidates, totalling 1620 fits
Best XGBoost params: {'gamma': 0, 'learning_rate': 0.02, 'max_depth': 3, 'n_estimators': 500, 'subsample': 0.8}
Best XGBoost MAE: 76156.925551525

===== Processing weekly dataset =====
Running AdaBoost GridSearchCV for weekly...
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best AdaBoost params: {'learning_rate': 0.04, 'n_estimators': 150}
Best AdaBoost MAE: 477893.76497900934

Running XGBoost GridSearchCV for weekly...
Fitting 5 folds for each of 324 candidates, totalling 1620 fits


KeyboardInterrupt: 